# World Cup Predictor Model Validation

This notebook audits leakage safeguards and runs match-level walk-forward backtests for the 2014, 2018, and 2022 World Cups. Historical tournaments validate match predictions only; they do not validate the 48-team World Cup 2026 bracket.

## 1. Setup

In [ ]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path

import pandas as pd

REPO_URL = "https://github.com/wdqgallego-git/worldcup-predictor.git"
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "src").exists():
    PROJECT_DIR = Path("/content/worldcup-predictor")
    if not PROJECT_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
for module_dir in (PROJECT_DIR / "src", PROJECT_DIR / "scripts"):
    if str(module_dir) not in sys.path:
        sys.path.insert(0, str(module_dir))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
AUDIT_OUTPUT_DIR = Path(tempfile.mkdtemp(prefix="worldcup_validation_"))
print(f"Project directory: {PROJECT_DIR}")
print(f"Temporary audit outputs: {AUDIT_OUTPUT_DIR}")

## 2. Download/load data

Set `REFRESH_DATA = True` to download fresh raw data. Existing manifest-backed files are loaded by default.

In [ ]:
from data_loader import load_rankings, load_results
from download_data import download_all

REFRESH_DATA = False
if REFRESH_DATA:
    download_all()

results = load_results()
rankings = load_rankings()
print(f"Historical matches: {len(results):,}")
print(f"Ranking rows: {len(rankings):,}")
results.head()

## 3. Leakage checks

The feature table must use only information available before each match. The checks below fail loudly for future matches, ranking-date leakage, target columns, placeholder features, NaN values, or infinite values.

In [ ]:
from features import build_training_table
from leakage_checks import (
    check_no_future_matches_used,
    check_no_missing_feature_values,
    check_no_placeholder_features,
    check_no_target_columns_in_features,
    check_ranking_dates_before_match,
)

AUDIT_CUTOFFS = {
    2014: "2014-06-01",
    2018: "2018-06-01",
    2022: "2022-11-01",
}
training_df, feature_cols = build_training_table(results, rankings)
check_no_target_columns_in_features(feature_cols)
check_no_placeholder_features(feature_cols)
check_no_missing_feature_values(training_df, feature_cols)
check_ranking_dates_before_match(training_df)

cutoff_rows = []
for year, cutoff in AUDIT_CUTOFFS.items():
    historical_training = training_df[training_df["date"] < pd.Timestamp(cutoff)].copy()
    check_no_future_matches_used(historical_training, cutoff)
    cutoff_rows.append({"year": year, "cutoff": cutoff, "training_rows": len(historical_training), "passed": True})

pd.DataFrame(cutoff_rows)

## 4. Match backtesting

Each historical World Cup contributes exactly 64 held-out match rows. Backtest outputs are saved to the temporary directory created during setup.

In [ ]:
from backtesting import run_walk_forward_backtests

backtest_outputs = run_walk_forward_backtests(output_dir=AUDIT_OUTPUT_DIR)
backtest_predictions = backtest_outputs["backtest_predictions"]
backtest_metrics = backtest_outputs["backtest_metrics"]
baseline_comparison = backtest_outputs["backtest_baseline_comparison"]

matches_by_year = backtest_predictions.groupby("backtest_year").size().rename("matches")
assert matches_by_year.eq(64).all(), "Every historical World Cup must contain exactly 64 test matches."
matches_by_year.to_frame()

## 5. Model comparison

Candidate goal models are evaluated on a chronological validation split rather than training fit.

In [ ]:
from backtesting import compare_goal_models

model_comparison = compare_goal_models(training_df=training_df, feature_cols=feature_cols)
model_comparison.to_csv(AUDIT_OUTPUT_DIR / "backtest_goal_model_comparison.csv", index=False)
model_comparison

## 6. Independent Poisson vs Dixon-Coles

In [ ]:
poisson_comparison = backtest_metrics[
    [
        "year",
        "model_name",
        "probability_method",
        "matches",
        "total_challenge_points",
        "average_challenge_points",
        "exact_score_rate",
        "correct_result_rate",
        "correct_goal_difference_rate",
    ]
].copy()
assert set(poisson_comparison["probability_method"]) == {"independent", "dixon_coles"}
poisson_comparison

## 7. Baseline comparison

In [ ]:
REQUIRED_BASELINES = {
    "always_1_1",
    "favorite_1_0",
    "favorite_2_0_if_strong",
    "most_likely_poisson",
    "expected_points_optimized",
}
assert REQUIRED_BASELINES.issubset(set(baseline_comparison["baseline"]))
baseline_overall = baseline_comparison.groupby("baseline", as_index=False).agg(
    total_challenge_points=("total_challenge_points", "sum"),
    tournaments=("year", "size"),
)
baseline_overall["matches"] = baseline_overall["tournaments"] * 64
baseline_overall["avg_challenge_points"] = (
    baseline_overall["total_challenge_points"] / baseline_overall["matches"]
)
baseline_overall = baseline_overall.sort_values("avg_challenge_points", ascending=False, kind="stable")
baseline_overall.to_csv(AUDIT_OUTPUT_DIR / "backtest_baseline_overall.csv", index=False)
baseline_overall

## 8. Readiness checklist

In [ ]:
expected_output_files = [
    "backtest_predictions.csv",
    "backtest_metrics.csv",
    "backtest_baseline_comparison.csv",
    "backtest_goal_model_comparison.csv",
    "backtest_baseline_overall.csv",
]
readiness = pd.DataFrame(
    [
        {"check": "Leakage checks pass", "passed": True},
        {"check": "2014, 2018, and 2022 each have 64 held-out matches", "passed": matches_by_year.eq(64).all()},
        {"check": "Independent Poisson evaluated", "passed": "independent" in set(poisson_comparison["probability_method"])},
        {"check": "Dixon-Coles evaluated", "passed": "dixon_coles" in set(poisson_comparison["probability_method"])},
        {"check": "Requested baselines evaluated", "passed": REQUIRED_BASELINES.issubset(set(baseline_comparison["baseline"]))},
        {"check": "Temporary audit reports generated", "passed": all((AUDIT_OUTPUT_DIR / name).exists() for name in expected_output_files)},
    ]
)
assert readiness["passed"].all(), "Validation readiness checklist failed."
print(f"Temporary validation outputs: {AUDIT_OUTPUT_DIR}")
print("Model validation readiness checklist passed.")
readiness